# Playback OpenSim Mot file
In this notebook is shown how to load a mot file generate by OpenSim and play it back on the MyoSkeleton

In [ ]:
import os
import time
from urllib.parse import urlparse

import mujoco
import numpy as np
import pandas as pd
import requests
import imageio
from IPython.display import Video, display

def read_mot(filename):
    # --- Download from URL if needed ---
    if filename.startswith("http://") or filename.startswith("https://"):
        parsed_url = urlparse(filename)
        local_path = os.path.basename(parsed_url.path) or "downloaded_file.mot"
        if not os.path.exists(local_path):
            try:
                req = requests.get(filename, allow_redirects=True)
                req.raise_for_status()
                with open(local_path, "wb") as f:
                    f.write(req.content)
                print(f"File '{local_path}' downloaded successfully.")
            except requests.exceptions.RequestException as e:
                print(f"Error downloading file: {e}")
                raise
        filename = local_path
    elif not os.path.exists(filename):
        raise FileNotFoundError(filename)
    # Initialize a counter for the number of rows to skip
    skiprows = 0

    # Open the file and read line by line
    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            # Increment the skiprows counter for each line until "endheader" is found
            if line.strip() == "endheader":
                break
            skiprows += 1

    # Now read the CSV file, skipping the determined number of rows
    df = pd.read_csv(filename, sep="\t", skiprows=skiprows + 1)

    return df

In [ ]:
url = "https://raw.githubusercontent.com/opensim-org/opensim-models/refs/heads/master/Pipelines/Gait2392_Simbody/OutputReference/subject01_walk1_ik.mot"
df = read_mot(url)

mj_model = mujoco.MjModel.from_xml_path(
    "../myosuite/simhive/myo_model/myoskeleton/myoskeleton.xml"
)
mj_data = mujoco.MjData(mj_model)

joint_names = [mj_model.joint(jn).name for jn in range(mj_model.njnt)]
subc = [c for c in df.columns if c in joint_names]

print(
    f"Joints in the Mot files that are not present in the MJC model: {set(subc) - set(joint_names)}"
)

# ---- camera settings
camera = mujoco.MjvCamera()
camera.azimuth = 90
camera.distance = 3
camera.elevation = -45.0
camera.lookat = [0,0,.75]
options_ref = mujoco.MjvOption()
options_ref.flags[:] = 0
options_ref.geomgroup[1:] = 0
renderer_ref = mujoco.Renderer(mj_model)
renderer_ref.scene.flags[:] = 0
frames=[]
from tqdm import tqdm
for t in tqdm(range(len(df)), desc="Rendering frames"):
    for jn in subc:
        mjc_j_idx = mj_model.joint(joint_names.index(jn)).qposadr
        mj_data.qpos[mjc_j_idx] = np.deg2rad(df[jn].loc[t])
        if "knee_angle" in jn:  # knee joints have negative sign in myosuite
            mj_data.qpos[mjc_j_idx] *= -1

    mujoco.mj_forward(mj_model, mj_data)
    renderer_ref.update_scene(mj_data, camera=camera)#, scene_option=options_ref)
    frame = renderer_ref.render()
    frames.append(frame)

os.makedirs('videos_test', exist_ok = True)
output_name = 'videos_test/playback_mot.mp4'
imageio.mimwrite(output_name, frames, fps=60.0)
# show in the notebook
display(Video(output_name, embed=True))